In [ ]:
# ===========================================================
#   PAN-STARRS (PS1) BULK FITS DOWNLOADER — FINAL STABLE VERSION
#   Fully compatible with Python 3.6+ (NO walrus operator)
#   Handles:
#       • display?image=... links
#       • multi-extension FITS (HDU[1], HDU[2], ...)
#       • relative URLs, missing schemes
#       • retry, debug HTML saving
# ===========================================================

import os, time, urllib.parse, re, shutil
import requests
from bs4 import BeautifulSoup
from astropy.io import fits

# ---------------- CONFIG ----------------
OUT_DIR = "ps1_fits"
DEBUG_DIR = os.path.join(OUT_DIR, "debug_html")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

BASE_CGI = "https://ps1images.stsci.edu/cgi-bin/ps1cutouts"
BASE_HOST = "https://ps1images.stsci.edu"
HEADERS = {"User-Agent": "Mozilla/5.0 PS1Downloader/1.2"}

SIZE = 240            # arcsec cutout size
FILTERS = ["g","r","i","z","y"]
SLEEP = 0.5           # polite delay between downloads

# ---------------- SUPERNOVAE LIST (RA, DEC in degrees) ----------------
# Add all 27 SNe here:
STARS = {
    #"SN2004A":    (163.39383, 30.55228),
    #"SN2004et":   (180.33083, 60.55103),
    #"SN2005cs":   (202.51358, 47.17658),
    #"SN2006my":   (178.86225, 67.89392),
    #"SN2006ov":   (184.23617, 38.77586),
    #"SN2008ax":   (147.09300, 41.24578),
    #"SN2009hd":   (150.21721, 41.66206),
    #"SN2011dh":   (187.70467, 47.17436),
    #"SN2012A":    (136.48767, -6.81131),
    #"SN2012aw":   (151.67417, 11.91520),
    #"SN2012ec":   (338.94304, -0.30789),
    #"SN2013ej":   (23.56692, 30.65694),
    #"SN2016gkg":  (12.20250, -29.19944),
    #"SN2017eaw":  (308.71971, 60.16019),
    #"SN2018aoz":  (181.80292, 71.35225),
    #"SN2019yvr":  (45.57700, -8.63000),
    #"iPTF13bvn":  (182.53417, 36.20436),
    "SN2007gr":   (157.32100, 55.80283),
    "SN2005gl":   (38.42400, 18.59300),
    "SN2015bs":   (154.40179, 25.29189),
    "SN2016aqf":  (132.33040, 50.04640),
    "SN2020fqv":  (187.72508, 47.17847),
    "SN2023ixf":  (210.80254, 54.34861),
    # Add more if needed...
}

# ---------------- helpers ----------------

def is_fits_bytes(b):
    """Quick signature check."""
    try:
        return b.startswith(b"SIMPLE")
    except:
        return False


def normalize_candidate_url(url):
    """Convert a raw string/relative URL into absolute PS1 FITS URL."""
    if not url:
        return None

    url = url.strip()
    url = urllib.parse.unquote(url)
    url = url.replace("&amp;", "&")

    # Case: viewer link with ?image=/path/to/file.fits
    if "image=" in url:
        try:
            parsed = urllib.parse.urlparse(url)
            q = urllib.parse.parse_qs(parsed.query)
            for key in ["image", "img", "file"]:
                if key in q:
                    img = q[key][0]
                    if img.startswith("http://") or img.startswith("https://"):
                        return img
                    if img.startswith("/"):
                        return urllib.parse.urljoin(BASE_HOST, img)
                    return urllib.parse.urljoin(BASE_HOST + "/", img)
        except:
            pass

    # Full URL
    if url.startswith("http://") or url.startswith("https://"):
        return url

    # Protocol-relative
    if url.startswith("//"):
        return "https:" + url

    # Absolute path on PS1 server
    if url.startswith("/"):
        return urllib.parse.urljoin(BASE_HOST, url)

    # Bare FITS filename or rings.v3... etc.
    if url.endswith(".fits") or url.startswith("rings"):
        return urllib.parse.urljoin(BASE_HOST + "/", url)

    return None


def save_debug(name, content, mode="w"):
    """Save debug HTML/bin."""
    path = os.path.join(DEBUG_DIR, name)
    if "b" in mode:
        with open(path, mode) as f:
            f.write(content)
    else:
        with open(path, mode, encoding="utf-8") as f:
            f.write(content)
    return path


def validate_and_save_fits(tmp_path, out_path):
    """
    PS1 FITS images are usually in HDU[1] (NOT HDU[0]).
    This function searches ALL HDUs for the first 2D image.
    """
    try:
        hdul = fits.open(tmp_path, ignore_missing_end=True)
    except Exception as e:
        return False, str(e)

    good_hdu = None
    for h in hdul:
        data = h.data
        if data is None:
            continue
        if hasattr(data, "shape") and len(data.shape) == 2:
            good_hdu = h
            break

    if good_hdu is None:
        hdul.close()
        return False, "no 2D image found in any HDU"

    # write clean FITS containing only the good HDU
    fits.writeto(out_path, good_hdu.data, good_hdu.header, overwrite=True)
    hdul.close()
    return True, None


# ---------------- download loop ----------------

for sn, (ra, dec) in STARS.items():
    print("\n========================================")
    print("Star:", sn, "| RA:", ra, " DEC:", dec)
    print("========================================")

    for flt in FILTERS:
        print("  → Filter:", flt)

        params = {
            "pos": f"{ra}+{dec}",
            "filter": flt,
            "size": str(SIZE),
            "format": "fits",
            "filetypes": "stack"
        }

        # Fetch HTML selector page
        try:
            r = requests.get(BASE_CGI, params=params, headers=HEADERS, timeout=60)
        except Exception as e:
            print("    CGI request error:", e)
            continue

        if r.status_code != 200:
            print("    CGI returned HTTP", r.status_code)
            save_debug(f"{sn}_{flt}_cgi_error.html", r.text)
            continue

        html = r.text
        soup = BeautifulSoup(html, "html.parser")

        # Collect ALL potential .fits references
        candidates = []

        # 1) href= links
        for a in soup.find_all("a", href=True):
            if ".fits" in a["href"]:
                candidates.append(a["href"])

        # 2) attributes
        for tag in soup.find_all(True):
            for attr in ["data-href","data-url","onclick","onmousedown","value","data-file"]:
                v = tag.get(attr)
                if v and ".fits" in v:
                    candidates.append(v)

        # 3) raw text
        text_chars = soup.get_text(" ")
        for m in re.findall(r"[^\s\"']+\.fits[^\s\"']*", text_chars, re.IGNORECASE):
            candidates.append(m)

        # 4) regex on raw HTML
        for m in re.findall(r"(/?[\w\.\-\/]+\.fits(?:\?[^\"'\s>]*)?)", html, re.IGNORECASE):
            candidates.append(m)

        # unique ordered
        final_candidates = []
        seen = set()
        for c in candidates:
            if c not in seen:
                seen.add(c)
                final_candidates.append(c)

        if not final_candidates:
            print("    No candidate FITS URLs → saving debug")
            save_debug(f"{sn}_{flt}_no_candidates.html", html)
            continue

        # Try each candidate until a valid FITS is found
        saved_ok = False

        for cand in final_candidates:
            fits_url = normalize_candidate_url(cand)
            if not fits_url:
                continue

            print("    candidate:", fits_url)

            # Attempt direct fetch
            try:
                r2 = requests.get(fits_url, headers=HEADERS, timeout=120)
            except Exception as e:
                print("      fetch error:", e)
                continue

            content_type = r2.headers.get("Content-Type", "").lower()

            # Viewer page HTML
            if "text/html" in content_type:
                inner_html = r2.text

                # Look for ?image=...fits
                m = re.search(r"[?&]image=([^&\"'>]+\.fits)", inner_html)
                if m:
                    extracted = urllib.parse.unquote(m.group(1))
                    if extracted.startswith("/"):
                        raw_url = urllib.parse.urljoin(BASE_HOST, extracted)
                    else:
                        raw_url = urllib.parse.urljoin(BASE_HOST + "/", extracted)

                    print("      extracted image param:", raw_url)

                    # fetch extracted FITS
                    try:
                        r3 = requests.get(raw_url, headers=HEADERS, timeout=120)
                    except Exception as e:
                        print("      extracted fetch error:", e)
                        continue

                    if r3.status_code == 200 and is_fits_bytes(r3.content):
                        tmp_path = os.path.join(OUT_DIR, f"{sn}_{flt}.tmp")
                        with open(tmp_path, "wb") as f:
                            f.write(r3.content)
                        out_path = os.path.join(OUT_DIR, f"{sn}_{flt}.fits")

                        valid, note = validate_and_save_fits(tmp_path, out_path)
                        if valid:
                            print("      ✔ Saved", out_path)
                            saved_ok = True
                            break
                        else:
                            print("      invalid FITS:", note)
                            save_debug(f"{sn}_{flt}_invalid_extracted.bin", r3.content, "wb")
                            continue

                # save viewer HTML
                save_debug(f"{sn}_{flt}_viewer.html", inner_html)
                continue

            # Direct binary
            data = r2.content
            if r2.status_code == 200 and is_fits_bytes(data):
                tmp_path = os.path.join(OUT_DIR, f"{sn}_{flt}.tmp")
                with open(tmp_path, "wb") as f:
                    f.write(data)
                out_path = os.path.join(OUT_DIR, f"{sn}_{flt}.fits")

                valid, note = validate_and_save_fits(tmp_path, out_path)
                if valid:
                    print("      ✔ Saved", out_path)
                    saved_ok = True
                    break
                else:
                    print("      invalid FITS:", note)
                    save_debug(f"{sn}_{flt}_invalid.bin", data, "wb")
                    continue
            else:
                save_debug(f"{sn}_{flt}_bad_response.bin", data, "wb")
                continue

        if not saved_ok:
            print("    ✖ No valid FITS found for", sn, flt)

        time.sleep(SLEEP)

print("\n===================================")
print("DONE — All FITS saved in:", OUT_DIR)
print("===================================")


Star: SN2007gr | RA: 157.321  DEC: 55.80283
  → Filter: g
    candidate: https://ps1images.stsci.edu/rings.v3.skycell/2373/094/rings.v3.skycell.2373.094.stk.g.unconv.fits
      ✔ Saved ps1_fits/SN2007gr_g.fits
  → Filter: r
    candidate: https://ps1images.stsci.edu/rings.v3.skycell/2373/094/rings.v3.skycell.2373.094.stk.r.unconv.fits
      ✔ Saved ps1_fits/SN2007gr_r.fits
  → Filter: i
    candidate: https://ps1images.stsci.edu/rings.v3.skycell/2373/094/rings.v3.skycell.2373.094.stk.i.unconv.fits
      ✔ Saved ps1_fits/SN2007gr_i.fits
  → Filter: z
    candidate: https://ps1images.stsci.edu/rings.v3.skycell/2373/094/rings.v3.skycell.2373.094.stk.z.unconv.fits
      ✔ Saved ps1_fits/SN2007gr_z.fits
  → Filter: y
    candidate: https://ps1images.stsci.edu/rings.v3.skycell/2373/094/rings.v3.skycell.2373.094.stk.y.unconv.fits
      ✔ Saved ps1_fits/SN2007gr_y.fits

Star: SN2005gl | RA: 38.424  DEC: 18.593
  → Filter: g
    candidate: https://ps1images.stsci.edu/rings.v3.skycell/1687/063/

In [ ]:
import pandas as pd
import re

# ---------- 1. Helper name-normalizer ----------
def normalize_name(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    # remove punctuation except * and alphanumeric
    x = re.sub(r"[^\w\s\*]+", "", x)
    # replace multiple spaces with single
    x = re.sub(r"\s+", " ", x)
    return x.strip()

# ---------- 2. Load the base catalog ----------
df = pd.read_csv("RSG_and_close_stars_catalog_4_11_24.csv")

# Normalize the catalog names
df["name_norm"] = df["Alias"].astype(str).apply(normalize_name)

# ---------- 3. List of special progenitor stars ----------
special_stars_raw = [
    "VY CMa",
    "AH Sco",
    "VX Sgr",
    "mu. Cep",
    "S Per",
    "NML Cyg",
    "V869 Ara",
    "alf Ori",
    "alf Sco",
    "lam Vel",
    "psi01 Aur",
    "zet Cep",
    "eps Peg",
    "tet Del",
    "[W61] R 68",
    "[W61] R 71",
    "V* DU Cru",
    "V* V1064 Sco",
    "V* V1061 Sco",
    "V* V407 Pup",
    "V* V362 Nor",
    "V* AO Cru",
    "V* RX Tel",
]

# Normalize the special star names
special_norm = {normalize_name(x) for x in special_stars_raw}

# ---------- 4. Create the label ----------
df["special_label"] = df["name_norm"].apply(lambda x: 1 if x in special_norm else 0)

# ---------- 5. Save updated catalog ----------
df.to_csv("RSG_catalog_with_special_labels.csv", index=False)

print("Done. Special stars labeled.")
print(df["special_label"].value_counts())
print(df[df["special_label"] == 1][["Alias", "name_norm"]])

Done. Special stars labeled.
special_label
0    641
1     10
Name: count, dtype: int64
            Alias     name_norm
512   V* V407 Pup   v* v407 pup
518   V* V362 Nor   v* v362 nor
532    [W61] R 68      w61 r 68
533    [W61] R 71      w61 r 71
602  V* V1064 Sco  v* v1064 sco
619     V* AO Cru     v* ao cru
629     V* DU Cru     v* du cru
635     V* RX Tel     v* rx tel
639  V* V1061 Sco  v* v1061 sco
650       NML Cyg       nml cyg


In [ ]:
# preprocess_rsg_catalog.py
import pandas as pd
import numpy as np
import re, os

IN_CSV = "RSG_catalog_with_special_labels.csv"
OUT_CSV = "RSG_preprocessed.csv"
DROP_MISSING_THRESH = 0.80  # drop columns with >80% missing
DUP_DROP = True             # drop duplicate Alias rows

# Recommended core columns to keep if present
CORE_COLS = [
    "Alias", "ra", "dec", "parallax", "parallax_error", "pmra", "pmdec",
    "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag",
    "bp_rp", "bp_g", "g_rp",
    "MKs", "j_m", "h_m", "k_m",
    "A_Ks", "A_J", "A_H", "A_V",
    "r_med_geo", "r_lo_geo", "r_hi_geo",
    "Teff", "Teff_error", "log_Teff", "log(L/L.)", "Mbol", "mass_loss",
    "label", "image_source", "sn_label", "synthetic_sn",
    "progenitor_flag", "progenitor_priority", "special_label"
]

def is_img_col(c):
    return c.startswith("img_") or c.startswith("imgimg_")

def is_emb_col(c):
    # f0..f511 style
    return re.match(r"^f\d+$", str(c)) is not None

# --- load ---
if not os.path.exists(IN_CSV):
    raise SystemExit(f"Input CSV not found: {IN_CSV}")

df = pd.read_csv(IN_CSV)
print("Loaded:", IN_CSV, "shape:", df.shape)

# --- drop columns with too many missing values ---
miss_frac = df.isna().mean()
cols_to_drop_missing = miss_frac[miss_frac > DROP_MISSING_THRESH].index.tolist()
if cols_to_drop_missing:
    print(f"Dropping {len(cols_to_drop_missing)} columns with >{int(DROP_MISSING_THRESH*100)}% missing:")
    print(cols_to_drop_missing)
    df = df.drop(columns=cols_to_drop_missing)
else:
    print("No columns exceed missing threshold.")

# --- build keep list: core + img_* + f* present in df ---
present_core = [c for c in CORE_COLS if c in df.columns]
img_cols = [c for c in df.columns if is_img_col(c)]
emb_cols = [c for c in df.columns if is_emb_col(c)]
# also include any important numeric columns not in CORE but present (to be safe)
numeric_extra = [c for c in df.select_dtypes(include=[np.number]).columns if c not in present_core + img_cols + emb_cols and c != "index"]
# assemble final columns to keep
keep_cols = present_core + img_cols + emb_cols + numeric_extra

# ensure Alias is first
if "Alias" in keep_cols:
    keep_cols = ["Alias"] + [c for c in keep_cols if c != "Alias"]

# If image_source exists ensure we keep it
if "image_source" in df.columns and "image_source" not in keep_cols:
    keep_cols.append("image_source")

print("Columns to KEEP (sample 40):", keep_cols[:40])
print("Total keep columns:", len(keep_cols))

# prune dataframe to those columns (but keep Alias always)
df_keep = df[[c for c in keep_cols if c in df.columns]].copy()

# --- tidy Alias and remove duplicates if requested ---
df_keep['Alias'] = df_keep['Alias'].astype(str).str.strip()
if DUP_DROP:
    before = len(df_keep)
    df_keep = df_keep.drop_duplicates(subset=['Alias'], keep='first').reset_index(drop=True)
    after = len(df_keep)
    print(f"Dropped {before-after} duplicate Alias rows (kept first occurrence).")

# --- fill missing values ---
# numeric: median, object/string: "unknown" (except Alias)
num_cols = df_keep.select_dtypes(include=[np.number]).columns.tolist()
obj_cols = [c for c in df_keep.columns if c not in num_cols and c != 'Alias']

print("Numeric columns to median-impute (count):", len(num_cols))
print("Object columns to fill with 'unknown' (count):", len(obj_cols))

for c in num_cols:
    med = df_keep[c].median()
    if np.isnan(med):
        # if column entirely NaN, fill with 0
        med = 0.0
    df_keep[c] = df_keep[c].fillna(med)

for c in obj_cols:
    df_keep[c] = df_keep[c].fillna("unknown").astype(str)

# --- final checks and write out ---
print("Final preprocessed shape:", df_keep.shape)
# report remaining missingness (should be zero now except Alias)
missing_after = df_keep.isna().sum()
print("Missing after (non-zero):")
print(missing_after[missing_after > 0])

df_keep.to_csv(OUT_CSV, index=False)
print("Wrote preprocessed CSV:", OUT_CSV)

Loaded: RSG_catalog_with_special_labels.csv shape: (651, 179)
Dropping 46 columns with >80% missing:
['pf', 'pf_error', 'metallicity', 'metallicity_error', 'frequency', 'frequency_error', 'amplitude', 'nss_solution_type', 'period', 'period_error', 'eccentricity', 'eccentricity_error', 'center_of_mass_velocity', 'center_of_mass_velocity_error', 'flags', 'mass_ratio', 'mass_ratio_error', 'best_class_name_dr2', 'pf_dr2', 'pf_error_dr2', 'metallicity_dr2', 'metallicity_error_dr2', 'frequency_dr2', 'frequency_error_dr2', 'AAVSO_Type', 'AAVSO_max_mag', 'AAVSO_passband_max_mag', 'AAVSO_min_mag', 'AAVSO_passband_min_mag', 'AAVSO_Period', 'ASASSN-V', 'ASASSN_Vmag', 'ASASSN_Amp', 'ASASSN_Period', 'ASASSN_Type', 'ASASSN_Prob', 'Binary_details', 'Binary_Ref', 'Binary_Ref_#', 'Binary_Flag', 'B_field', 'B_field_Ref', 'B_field_Ref_#', 'Runaway', 'Runaway_Ref', 'Runaway_Ref_#']
Columns to KEEP (sample 40): ['Alias', 'ra', 'dec', 'parallax', 'parallax_error', 'pmra', 'pmdec', 'phot_g_mean_mag', 'phot_b

In [ ]:
# extract_features_pos_neg_recursive.py
# Robust extractor: recursively process all FITS files under listed directories,
# extract simple image features + ResNet18 embeddings, save csv.
#
# Output: cnn_features_pos_neg_full.csv

import os, sys
from glob import iglob
import numpy as np
import pandas as pd
from astropy.io import fits
from scipy import stats
from PIL import Image
import torch, torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models

# ---------- CONFIG ----------
FITS_DIRS = ["ps1_fits", "/content/drive/MyDrive/rsg_negative_fits"]   # directories to scan recursively
OUT_CSV = "cnn_features_pos_neg_full.csv"
PATCH_SIZE_PIX = 200
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERBOSE = True

# extensions to match (case-insensitive); supports gzipped fits too
EXTS = [".fits", ".fit", ".FITS", ".FIT", ".fits.gz", ".fit.gz", ".FITS.GZ"]

# ---------------- helpers ----------------
def find_all_fits(paths):
    files = []
    for d in paths:
        if not os.path.isdir(d):
            continue
        # recursive search for any of the extensions (case-insensitive)
        for root, _, filenames in os.walk(d):
            for fn in filenames:
                low = fn.lower()
                if any(low.endswith(ext.lower()) for ext in EXTS):
                    files.append(os.path.join(root, fn))
    return sorted(set(files))

def load_best_hdu(path):
    try:
        hdul = fits.open(path, ignore_missing_end=True)
    except Exception as e:
        # try reading from gz if name doesn't end with .gz (astropy should handle gz too)
        return None, None
    good = None
    for h in hdul:
        if h.data is None:
            continue
        if hasattr(h.data, "shape") and len(h.data.shape) == 2:
            good = h
            break
    if good is None:
        hdul.close()
        return None, None
    data = np.array(good.data, dtype=float)
    hdr = good.header
    hdul.close()
    return data, hdr

def center_crop_pad(data, size):
    h, w = data.shape
    cy, cx = h//2, w//2
    half = size//2
    y0, y1 = max(0, cy-half), min(h, cy+half)
    x0, x1 = max(0, cx-half), min(w, cx+half)
    cropped = data[y0:y1, x0:x1]
    if cropped.shape == (size, size):
        return np.nan_to_num(cropped, np.nanmedian(cropped))
    out = np.full((size, size), np.nan, dtype=float)
    sy, sx = cropped.shape
    oy = (size - sy)//2; ox = (size - sx)//2
    out[oy:oy+sy, ox:ox+sx] = cropped
    out = np.nan_to_num(out, np.nanmedian(out))
    return out

def extract_simple_features(patch):
    p = patch.copy().astype(float)
    flat = p.flatten()
    flat = flat[np.isfinite(flat)]
    if flat.size == 0:
        keys = ['img_mean','img_median','img_std','img_min','img_max','img_p10','img_p90',
                'img_skew','img_kurtosis','img_ap_flux_r10','img_ap_flux_r25','img_ap_flux_r50','img_ap_flux_r90',
                'img_top_bot_ratio','img_entropy']
        return {k: float('nan') for k in keys}
    feats = {}
    feats['img_mean'] = float(np.mean(flat))
    feats['img_median'] = float(np.median(flat))
    feats['img_std'] = float(np.std(flat))
    feats['img_min'] = float(np.min(flat))
    feats['img_max'] = float(np.max(flat))
    feats['img_p10'] = float(np.percentile(flat, 10))
    feats['img_p90'] = float(np.percentile(flat, 90))
    feats['img_skew'] = float(stats.skew(flat))
    feats['img_kurtosis'] = float(stats.kurtosis(flat))
    h, w = p.shape
    cy, cx = h/2.0, w/2.0
    Y, X = np.indices(p.shape)
    r = np.sqrt((X-cx)**2 + (Y-cy)**2)
    for r_frac, name in [(0.1,'r10'), (0.25,'r25'), (0.5,'r50'), (0.9,'r90')]:
        thresh = r_frac * max(h,w)/2.0
        mask = r <= thresh
        feats[f'img_ap_flux_{name}'] = float(np.nansum(p[mask]))
    top_mean = np.nanmean(p[:h//3,:]) if np.isfinite(p[:h//3,:]).any() else 0.0
    bot_mean = np.nanmean(p[-h//3:,:]) if np.isfinite(p[-h//3:,:]).any() else 0.0
    feats['img_top_bot_ratio'] = float((top_mean+1e-12)/(bot_mean+1e-12))
    hist, _ = np.histogram(flat, bins=32, density=True)
    hist = hist + 1e-12
    feats['img_entropy'] = float(-np.sum(hist * np.log(hist)))
    return feats

# ---- ResNet setup (512-d features) ----
transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
                       T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = nn.Identity()
resnet.eval()
resnet.to(DEVICE)

def fits_patch_to_pil(patch):
    arr = patch.copy().astype(float)
    arr = arr - np.nanmin(arr)
    if np.nanmax(arr) > 0:
        arr = arr / np.nanmax(arr)
    arr = (arr * 255).clip(0,255).astype(np.uint8)
    return Image.fromarray(arr).convert("RGB")

def extract_resnet_emb(pil_img):
    x = transform(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = resnet(x)
    return feat.cpu().numpy().flatten()

# ---------------- main ----------------
fits_files = find_all_fits(FITS_DIRS)
if len(fits_files) == 0:
    print("No FITS found in paths:", FITS_DIRS)
    sys.exit(1)

# summary by folder
counts = {}
for p in fits_files:
    parent = os.path.normpath(p).split(os.sep)[1] if len(os.path.normpath(p).split(os.sep))>1 else ""
    counts.setdefault(parent, 0)
    counts[parent] += 1
print("Found total FITS:", len(fits_files))
print("Counts by top-level folder (sample):", counts)

rows = []
processed = 0
for idx, path in enumerate(fits_files, 1):
    fname = os.path.basename(path)
    try:
        data, hdr = load_best_hdu(path)
        if data is None:
            if VERBOSE: print("skip (no 2D HDU):", path)
            continue
        patch = center_crop_pad(data, PATCH_SIZE_PIX)
    except Exception as e:
        if VERBOSE: print("error loading", path, e)
        continue

    # features
    img_feats = extract_simple_features(patch)
    try:
        pil = fits_patch_to_pil(patch)
        emb = extract_resnet_emb(pil)
    except Exception as e:
        if VERBOSE: print("embedding failed for", path, e)
        emb = None

    rec = {
        "file": os.path.relpath(path),
        "image_source": os.path.basename(path),
        "__img_key": os.path.splitext(os.path.basename(path))[0]
    }

    if emb is not None:
        for j, v in enumerate(emb):
            rec[f"f{j}"] = float(v)
    rec.update(img_feats)
    rows.append(rec)
    processed += 1
    if VERBOSE and (idx % 20 == 0 or idx == len(fits_files)):
        print(f"Processed {idx}/{len(fits_files)} files (saved {len(rows)} rows).")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("Wrote", OUT_CSV, "rows:", len(df))
print("Done. Processed files:", processed)

Found total FITS: 135
Counts by top-level folder (sample): {'content': 40, 'SN2004A_g.fits': 1, 'SN2004A_i.fits': 1, 'SN2004A_r.fits': 1, 'SN2004A_y.fits': 1, 'SN2004A_z.fits': 1, 'SN2004et_g.fits': 1, 'SN2004et_i.fits': 1, 'SN2004et_r.fits': 1, 'SN2004et_y.fits': 1, 'SN2004et_z.fits': 1, 'SN2005cs_g.fits': 1, 'SN2005cs_i.fits': 1, 'SN2005cs_r.fits': 1, 'SN2005cs_y.fits': 1, 'SN2005cs_z.fits': 1, 'SN2005gl_g.fits': 1, 'SN2005gl_i.fits': 1, 'SN2005gl_r.fits': 1, 'SN2005gl_y.fits': 1, 'SN2005gl_z.fits': 1, 'SN2006my_g.fits': 1, 'SN2006my_i.fits': 1, 'SN2006my_r.fits': 1, 'SN2006my_y.fits': 1, 'SN2006my_z.fits': 1, 'SN2006ov_g.fits': 1, 'SN2006ov_i.fits': 1, 'SN2006ov_r.fits': 1, 'SN2006ov_y.fits': 1, 'SN2006ov_z.fits': 1, 'SN2007gr_g.fits': 1, 'SN2007gr_i.fits': 1, 'SN2007gr_r.fits': 1, 'SN2007gr_y.fits': 1, 'SN2007gr_z.fits': 1, 'SN2008ax_g.fits': 1, 'SN2008ax_i.fits': 1, 'SN2008ax_r.fits': 1, 'SN2008ax_y.fits': 1, 'SN2008ax_z.fits': 1, 'SN2009hd_g.fits': 1, 'SN2009hd_i.fits': 1, 'SN200

In [ ]:
# step_generate_rsg_like_from_fits_optionB.py

import pandas as pd
import numpy as np
from astropy.io import fits
import os

CNN_CSV = "/content/cnn_features_pos_neg_full.csv"
RSG_CSV = "/content/RSG_preprocessed_augmented.csv"
OUT_CSV = "/content/RSG_image_like_rows_optionB.csv"

print("Loading CNN features...")
cnn = pd.read_csv(CNN_CSV)
rsg = pd.read_csv(RSG_CSV)

# -----------------------------
# 1. Extract RSG schema
# -----------------------------
rsg_columns = list(rsg.columns)

# These columns will be filled using image information
computable_cols = ["ra", "dec"]

# -----------------------------
# 2. Extract RA/DEC from FITS headers
# -----------------------------
def extract_radec_from_fits(path):
    try:
        hdul = fits.open(path, ignore_missing_end=True)
    except:
        return np.nan, np.nan

    ra = dec = np.nan
    for h in hdul:
        hdr = h.header

        # Try common header keywords
        for key_ra in ["RA", "CRVAL1", "RA_OBJ", "OBJCTRA"]:
            if key_ra in hdr:
                try:
                    ra = float(hdr[key_ra])
                    break
                except:
                    pass

        for key_dec in ["DEC", "CRVAL2", "DEC_OBJ", "OBJCTDEC"]:
            if key_dec in hdr:
                try:
                    dec = float(hdr[key_dec])
                    break
                except:
                    pass

        if not np.isnan(ra) and not np.isnan(dec):
            break

    hdul.close()
    return ra, dec


# -----------------------------
# 3. Build new synthetic RSG-like rows
# -----------------------------
rows = []
fits_dir_guess = []

# detect directories
for c in ["file", "image_source"]:
    if c in cnn.columns:
        fits_dir_guess.append(c)

if not fits_dir_guess:
    raise ValueError("No file/image_source column found in cnn csv.")

file_col = fits_dir_guess[0]

for i, row in cnn.iterrows():

    fits_path = row[file_col]

    if not os.path.isfile(fits_path):
        # maybe the file is only a filename, try drive/MyDrive
        base = os.path.basename(fits_path)
        alt = "/content/drive/MyDrive/" + base
        if os.path.isfile(alt):
            fits_path = alt
        else:
            fits_path = None

    # extract RA/DEC if possible
    ra, dec = (np.nan, np.nan)
    if fits_path and os.path.exists(fits_path):
        ra, dec = extract_radec_from_fits(fits_path)

    # build blank RSG row
    new_row = {col: np.nan for col in rsg_columns}

    # fill minimal computable fields
    new_row["ra"] = ra
    new_row["dec"] = dec
    new_row["special_label"] = np.nan   # we will assign synthetic labels later

    # keep image file reference
    new_row["source_image_file"] = row[file_col]

    # append image features
    for c in cnn.columns:
        if c not in rsg_columns:
            new_row[c] = row[c]

    rows.append(new_row)

# -----------------------------
# 4. Save output
# -----------------------------
df_new = pd.DataFrame(rows)
df_new.to_csv(OUT_CSV, index=False)

print("Wrote:", OUT_CSV)
print("Rows:", len(df_new))
print("Sample columns:", df_new.columns[:25].tolist())
print(df_new[["source_image_file", "ra", "dec"]].head())

Loading CNN features...
Wrote: /content/RSG_image_like_rows_optionB.csv
Rows: 135
Sample columns: ['Alias', 'ra', 'dec', 'parallax', 'parallax_error', 'pmra', 'pmdec', 'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp', 'bp_g', 'g_rp', 'MKs', 'j_m', 'h_m', 'k_m', 'A_Ks', 'A_J', 'A_H', 'A_V', 'r_med_geo', 'r_lo_geo', 'r_hi_geo', 'Teff']
                                source_image_file          ra   dec
0  drive/MyDrive/rsg_negative_fits/NML Cyg_g.fits  307.058807  42.0
1  drive/MyDrive/rsg_negative_fits/NML Cyg_i.fits  307.058807  42.0
2  drive/MyDrive/rsg_negative_fits/NML Cyg_r.fits  307.058807  42.0
3  drive/MyDrive/rsg_negative_fits/NML Cyg_y.fits  307.058807  42.0
4  drive/MyDrive/rsg_negative_fits/NML Cyg_z.fits  307.058807  42.0


In [ ]:
import pandas as pd

infile = "/content/RSG_image_like_rows_optionB_withCNN.csv"
outfile = "/content/RSG_image_like_rows_optionB_withCNN_labeled.csv"

df = pd.read_csv(infile)

def infer_label(path):
    path = str(path).lower()
    if "ps1_fits" in path:
        return 1
    if "rsg_negative_fits" in path:
        return 0
    return 0  # default negative if unknown

df["img_label"] = df["source_image_file"].apply(infer_label)

df.to_csv(outfile, index=False)

print("Wrote:", outfile)
print(df["img_label"].value_counts())

Wrote: /content/RSG_image_like_rows_optionB_withCNN_labeled.csv
img_label
1    95
0    40
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np

rsg = pd.read_csv("/content/RSG_preprocessed_augmented.csv")
cnn = pd.read_csv("/content/RSG_image_like_rows_optionB_withCNN_labeled.csv")

# Identify all CNN feature columns
cnn_fcols  = [c for c in cnn.columns if c.startswith("f")]
cnn_icols  = [c for c in cnn.columns if c.startswith("img_")]
cnn_extra  = cnn_fcols + cnn_icols

# Ensure RSG has these columns (filled with NaN)
for col in cnn_extra:
    if col not in rsg.columns:
        rsg[col] = np.nan

# Ensure CNN has the RSG target label name
cnn["special_label"] = cnn["img_label"]

# Columns to keep = RSG numeric + cnn features + target
keep = list(rsg.columns)

# Reorder CNN to same columns (missing ones will auto-fill with NaN)
cnn_final = cnn.reindex(columns=keep)

# Combine
full = pd.concat([rsg[keep], cnn_final], ignore_index=True)

out = "/content/model_training_dataset.csv"
full.to_csv(out, index=False)

print("Saved:", out)
print("Final shape:", full.shape)
print("Label counts:", full["special_label"].value_counts())


Saved: /content/model_training_dataset.csv
Final shape: (986, 634)
Label counts: special_label
0.0    681
1.0    305
Name: count, dtype: int64


In [ ]:
# leakfree_image_only_with_pca_rf.py
# - Per-fold PCA on embeddings (fit on train only)
# - RandomForest (no calibration inside CV)
# - Final model calibrated once after full training
# Inputs: /content/model_training_star_level.csv
# Outputs:
#  - /content/star_level_leakfree_image_only_pca_predictions.csv
#  - /content/sn_model_leakfree_image_only_pca.joblib

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve
import joblib
import warnings
warnings.filterwarnings("ignore")

INPUT = "/content/model_training_star_level.csv"
OUT_PRED = "/content/star_level_leakfree_image_only_pca_predictions.csv"
OUT_MODEL = "/content/sn_model_leakfree_image_only_pca.joblib"

RANDOM_STATE = 42
N_SPLITS = 8
PCA_DIM = 64        # reduce 512 -> 64 (tuneable)
RF_N_EST = 500
RF_MAX_DEPTH = None

print("Load:", INPUT)
df = pd.read_csv(INPUT)
print("Rows:", len(df))
print("Label dist:\n", df["special_label"].value_counts())

# feature selection: emb f0.. + img_*
emb_cols = sorted([c for c in df.columns if str(c).startswith("f")], key=lambda x:int(x[1:]) if x[1:].isdigit() else x)
img_cols = [c for c in df.columns if c.startswith("img_")]
print("Emb dims:", len(emb_cols), "img cols:", len(img_cols))

if len(emb_cols)==0:
    raise SystemExit("No embedding columns found (f0..). Abort.")

X_emb = df[emb_cols].copy()
X_img = df[img_cols].copy() if img_cols else pd.DataFrame(index=df.index)
y = df["special_label"].fillna(0).astype(int).values

# We'll do per-fold: imputer -> PCA on embeddings -> concat img features -> scaler -> RF
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_stats = []
best_thresholds = []

fold = 1
print("\nRunning StratifiedKFold CV (no calibration inside folds)...\n")
for tr_idx, val_idx in skf.split(X_emb, y):
    print("Fold", fold)
    Xe_tr = X_emb.iloc[tr_idx].values
    Xe_val = X_emb.iloc[val_idx].values
    Xi_tr = X_img.iloc[tr_idx].values if not X_img.empty else np.zeros((len(tr_idx),0))
    Xi_val = X_img.iloc[val_idx].values if not X_img.empty else np.zeros((len(val_idx),0))
    y_tr = y[tr_idx]; y_val = y[val_idx]

    # impute embeddings & img separately (median)
    imp_emb = SimpleImputer(strategy="median")
    imp_img = SimpleImputer(strategy="median") if Xi_tr.shape[1]>0 else None
    Xe_tr_imp = imp_emb.fit_transform(Xe_tr)
    Xe_val_imp = imp_emb.transform(Xe_val)
    if imp_img:
        Xi_tr_imp = imp_img.fit_transform(Xi_tr)
        Xi_val_imp = imp_img.transform(Xi_val)
    else:
        Xi_tr_imp = np.zeros((Xe_tr_imp.shape[0],0))
        Xi_val_imp = np.zeros((Xe_val_imp.shape[0],0))

    # PCA on embeddings (fit on train only)
    pca = PCA(n_components=min(PCA_DIM, Xe_tr_imp.shape[1]), random_state=RANDOM_STATE)
    Xe_tr_p = pca.fit_transform(Xe_tr_imp)
    Xe_val_p = pca.transform(Xe_val_imp)

    # concat reduced embeddings + img raw features
    Xtr = np.hstack([Xe_tr_p, Xi_tr_imp])
    Xval = np.hstack([Xe_val_p, Xi_val_imp])

    # scale numeric features
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xval_s = scaler.transform(Xval)

    # train RF (no calibration here)
    rf = RandomForestClassifier(n_estimators=RF_N_EST, max_depth=RF_MAX_DEPTH,
                                class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(Xtr_s, y_tr)

    probs_val = rf.predict_proba(Xval_s)[:,1]
    auc = roc_auc_score(y_val, probs_val) if len(np.unique(y_val))>1 else np.nan

    # baseline at 0.5
    preds50 = (probs_val >= 0.5).astype(int)
    acc50 = accuracy_score(y_val, preds50)
    prec50 = precision_score(y_val, preds50, zero_division=0)
    rec50 = recall_score(y_val, preds50, zero_division=0)
    f150 = f1_score(y_val, preds50, zero_division=0)

    # find best threshold by validation F1
    prec, rec, thr = precision_recall_curve(y_val, probs_val)
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thr[best_idx]) if best_idx < len(thr) else 0.5
    preds_best = (probs_val >= best_thr).astype(int)
    acc_best = accuracy_score(y_val, preds_best)
    prec_best = precision_score(y_val, preds_best, zero_division=0)
    rec_best = recall_score(y_val, preds_best, zero_division=0)
    f1_best = f1_score(y_val, preds_best, zero_division=0)

    print(f" @0.5 -> acc {acc50:.3f} prec {prec50:.3f} rec {rec50:.3f} f1 {f150:.3f} auc {auc:.3f}")
    print(f" best_thr {best_thr:.4f} -> acc {acc_best:.3f} prec {prec_best:.3f} rec {rec_best:.3f} f1 {f1_best:.3f}\n")

    fold_stats.append((acc50, prec50, rec50, f150, auc, acc_best, prec_best, rec_best, f1_best))
    best_thresholds.append(best_thr)
    fold += 1

# summarize CV
arr = np.array(fold_stats)
names = ["acc50","prec50","rec50","f150","auc","acc_best","prec_best","rec_best","f1_best"]
print("=== CV summary (mean ± std) ===")
for i,n in enumerate(names):
    print(f"{n:8s} : {arr[:,i].mean():.4f} ± {arr[:,i].std():.4f}")

avg_best_thr = float(np.nanmean(best_thresholds))
print("\nAverage best threshold across folds:", avg_best_thr)

# --------- Train final pipeline on FULL dataset ----------
print("\nTraining final model on full dataset (PCA on full embeddings) ...")
# full imputation
imp_full_emb = SimpleImputer(strategy="median").fit(X_emb.values)
Xe_full_imp = imp_full_emb.transform(X_emb.values)
imp_full_img = None
if not X_img.empty:
    imp_full_img = SimpleImputer(strategy="median").fit(X_img.values)
    Xi_full_imp = imp_full_img.transform(X_img.values)
else:
    Xi_full_imp = np.zeros((Xe_full_imp.shape[0],0))

# PCA on full
pca_full = PCA(n_components=min(PCA_DIM, Xe_full_imp.shape[1]), random_state=RANDOM_STATE)
Xe_full_p = pca_full.fit_transform(Xe_full_imp)

X_full = np.hstack([Xe_full_p, Xi_full_imp])

# scale
scaler_full = StandardScaler().fit(X_full)
X_full_s = scaler_full.transform(X_full)

# final RF (train on full)
final_rf = RandomForestClassifier(n_estimators=RF_N_EST, max_depth=RF_MAX_DEPTH,
                                  class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
final_rf.fit(X_full_s, y)

# calibrate final model with cross-val calibration (calibration done on full via CV)
calibrated = CalibratedClassifierCV(final_rf, method="isotonic", cv=5)
calibrated.fit(X_full_s, y)   # calibrated probabilities from cross-validation

probs_full = calibrated.predict_proba(X_full_s)[:,1]
preds_full = (probs_full >= avg_best_thr).astype(int)

# full metrics (training eval)
print("\nFinal (train-eval) metrics (after PCA + calibration):")
print("acc", accuracy_score(y, preds_full))
print("prec", precision_score(y, preds_full, zero_division=0))
print("rec", recall_score(y, preds_full, zero_division=0))
print("f1", f1_score(y, preds_full, zero_division=0))
print("auc", roc_auc_score(y, probs_full))
print("confusion:\n", confusion_matrix(y, preds_full))

# save outputs
out = df.copy()
out["prob_image_only_pca"] = probs_full
out["pred_image_only_pca"] = preds_full
out.to_csv(OUT_PRED, index=False)
print("\nWrote:", OUT_PRED)

joblib.dump({
    "pca": pca_full,
    "imp_emb": imp_full_emb,
    "imp_img": imp_full_img,
    "scaler": scaler_full,
    "model_calibrated": calibrated,
    "feature_emb_cols": emb_cols,
    "feature_img_cols": img_cols,
    "threshold": avg_best_thr
}, OUT_MODEL)
print("Saved model bundle to:", OUT_MODEL)
print("\nDone.")

Load: /content/model_training_star_level.csv
Rows: 986
Label dist:
 special_label
0    681
1    305
Name: count, dtype: int64
Emb dims: 512 img cols: 16

Running StratifiedKFold CV (no calibration inside folds)...

Fold 1
 @0.5 -> acc 0.782 prec 1.000 rec 0.289 f1 0.449 auc 0.661
 best_thr 0.4230 -> acc 0.339 prec 0.317 rec 1.000 f1 0.481

Fold 2
 @0.5 -> acc 0.798 prec 1.000 rec 0.359 f1 0.528 auc 0.706
 best_thr 0.6491 -> acc 0.798 prec 1.000 rec 0.359 f1 0.528

Fold 3
 @0.5 -> acc 0.789 prec 0.929 rec 0.342 f1 0.500 auc 0.675
 best_thr 0.6444 -> acc 0.789 prec 0.929 rec 0.342 f1 0.500

Fold 4
 @0.5 -> acc 0.748 prec 0.889 rec 0.211 f1 0.340 auc 0.619
 best_thr 0.4185 -> acc 0.341 prec 0.319 rec 1.000 f1 0.484

Fold 5
 @0.5 -> acc 0.748 prec 0.818 rec 0.237 f1 0.367 auc 0.616
 best_thr 0.4203 -> acc 0.325 prec 0.314 rec 1.000 f1 0.478

Fold 6
 @0.5 -> acc 0.797 prec 0.933 rec 0.368 f1 0.528 auc 0.688
 best_thr 0.6888 -> acc 0.805 prec 1.000 rec 0.368 f1 0.538

Fold 7
 @0.5 -> acc 0.8

In [ ]:
# map_cnn_to_rsg_by_alias_and_fuzzy.py
import os, re
import pandas as pd
import numpy as np
from difflib import get_close_matches
from collections import defaultdict

# ---------- CONFIG ----------
RSG_CSV = "/content/RSG_and_close_stars_catalog_4_11_24.csv"   # set to your original RSG master CSV path
CNN_CSV = "/content/cnn_features_pos_neg_full.csv"            # CNN features (embeddings) CSV
OUT_MAPPING = "/content/cnn_to_rsg_mapping_debug.csv"
OUT_UNMAPPED = "/content/unmapped_cnn_files.csv"
OUT_RSG_WITH_EMB = "/content/RSG_with_cnn_by_alias.csv"
OUT_RSG_ROWS_WITH_EMB = "/content/rsg_rows_with_embeddings.csv"

MIN_FUZZY_CUTOFF = 0.65   # tuneable
MAX_CLOSE_MATCHES = 3
VERBOSE = True
# ----------------------------

def basename_no_ext(x):
    if pd.isna(x): return ""
    b = os.path.basename(str(x))
    for ext in ['.fits.gz','.fit.gz','.fits','.fit','.FITS','.FIT','.gz','.png','.jpg','.jpeg']:
        if b.lower().endswith(ext.lower()):
            b = b[: -len(ext)]
    b = b.split("__")[0]
    return b

def normalize_text(s):
    if pd.isna(s): return ""
    s = str(s).lower().strip()
    # replace punctuation with spaces, keep alnum and spaces and star char
    s = re.sub(r"[^\w\s\*]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def token_set(x):
    x = normalize_text(x)
    toks = [t for t in re.split(r"\s+", x) if t]
    return set(toks)

# ---- load files ----
print("Loading inputs...")
rsg = pd.read_csv(RSG_CSV)
cnn = pd.read_csv(CNN_CSV)
print("rsg rows:", len(rsg), "cnn rows:", len(cnn))

# ---- prepare RSG alias normalizations ----
if 'Alias' not in rsg.columns:
    print("WARNING: RSG CSV does not contain an 'Alias' column. The script will create alias_norm empty.")
    rsg['Alias'] = rsg.get('Alias', "")

rsg['alias_norm'] = rsg['Alias'].astype(str).map(normalize_text)
# dedupe alias_norm -> first matching rsg index
alias_to_index = {}
for idx, nm in rsg['alias_norm'].items():
    if nm and nm not in alias_to_index:
        alias_to_index[nm] = idx

alias_list = sorted(list(set([a for a in rsg['alias_norm'].astype(str).unique() if a and a!="nan"])))
if VERBOSE:
    print("Unique non-empty alias_norm count:", len(alias_list))

# ---- prepare CNN filename normalizations ----
# detect file/path col
file_col = None
for c in ['file','image_source','image','image_file','source_file']:
    if c in cnn.columns:
        file_col = c
        break
# fallback: find any column that looks like path or endswith .fits in sample
if file_col is None:
    for c in cnn.columns:
        sample_vals = cnn[c].astype(str).head(30).tolist()
        if any(".fits" in v for v in sample_vals) or any("/" in v for v in sample_vals):
            file_col = c
            break

if file_col is None:
    print("WARNING: Could not find a filename column in CNN CSV. Creating 'file' from index.")
    cnn['file'] = ["cnn_file_%d" % i for i in range(len(cnn))]
    file_col = 'file'

cnn['file_basename'] = cnn[file_col].astype(str).map(basename_no_ext)
cnn['file_norm'] = cnn['file_basename'].astype(str).map(normalize_text)
cnn['file_tokens'] = cnn['file_norm'].map(lambda s: set(s.split()) if s else set())

if VERBOSE:
    print("Detected file column:", file_col)
    print("Sample cnn basenames:", cnn['file_basename'].head(6).tolist())

# ---- mapping strategies ----
# 1) exact basename match (case-insensitive)
# 2) exact normalized text match: file_norm == alias_norm
# 3) fuzzy match between file_norm and alias_list
# 4) token-overlap heuristic (alias tokens subset of file tokens or vice-versa)
# will record method and score

alias_norm_set = set(alias_list)

mapping_rows = []
mapped_cnn_idx = set()
mapped_rsg_idx = set()

# fast lookups
alias_norm_to_rsg_idxs = defaultdict(list)
for i, nm in rsg['alias_norm'].items():
    if nm and nm!="nan":
        alias_norm_to_rsg_idxs[nm].append(i)

# build reversed map for exact basename->alias_norm if any alias_norm equals basename
basename_to_alias = {}
for nm in alias_list:
    # if nm equals a basename in cnn, record (rare)
    basename_to_alias[nm] = nm if nm in cnn['file_basename'].values else None

# strategy 1: exact basename -> Alias (if identical)
for i_c, crow in cnn.iterrows():
    fb = crow['file_basename']
    if fb in rsg['Alias'].astype(str).values or normalize_text(fb) in alias_norm_to_rsg_idxs:
        # map to first rsg index with that alias_norm
        nm = normalize_text(fb)
        candidates = alias_norm_to_rsg_idxs.get(nm, [])
        if candidates:
            r_idx = candidates[0]
            mapping_rows.append({
                "cnn_index": i_c, "file": crow[file_col], "file_basename": fb,
                "method": "exact_basename", "rsg_index": r_idx, "rsg_alias": rsg.at[r_idx,'Alias'],
                "score": 1.0
            })
            mapped_cnn_idx.add(i_c)
            mapped_rsg_idx.add(r_idx)

# strategy 2: exact normalized string equality (file_norm == alias_norm)
for i_c, crow in cnn.iterrows():
    if i_c in mapped_cnn_idx: continue
    fn = crow['file_norm']
    if fn and fn in alias_norm_to_rsg_idxs:
        r_idx = alias_norm_to_rsg_idxs[fn][0]
        mapping_rows.append({
            "cnn_index": i_c, "file": crow[file_col], "file_basename": crow['file_basename'],
            "method": "exact_norm", "rsg_index": r_idx, "rsg_alias": rsg.at[r_idx,'Alias'],
            "score": 0.99
        })
        mapped_cnn_idx.add(i_c)
        mapped_rsg_idx.add(r_idx)

# strategy 3: fuzzy matching file_norm -> alias_list
alias_candidates = alias_list[:]  # list for get_close_matches
for i_c, crow in cnn.iterrows():
    if i_c in mapped_cnn_idx: continue
    fn = crow['file_norm']
    if not fn: continue
    matches = get_close_matches(fn, alias_candidates, n=MAX_CLOSE_MATCHES, cutoff=MIN_FUZZY_CUTOFF)
    if matches:
        match = matches[0]
        r_idx = alias_norm_to_rsg_idxs.get(match,[None])[0]
        if r_idx is not None:
            mapping_rows.append({
                "cnn_index": i_c, "file": crow[file_col], "file_basename": crow['file_basename'],
                "method": "fuzzy", "rsg_index": r_idx, "rsg_alias": rsg.at[r_idx,'Alias'],
                "score": round( (len(set(fn.split()) & set(match.split()))/ (1+len(set(fn.split())|set(match.split()))) ) , 3)
            })
            mapped_cnn_idx.add(i_c)
            mapped_rsg_idx.add(r_idx)

# strategy 4: token overlap heuristics
for i_c, crow in cnn.iterrows():
    if i_c in mapped_cnn_idx: continue
    f_toks = crow['file_tokens']
    if not f_toks:
        continue
    best_r = None; best_score = 0.0
    # loop through alias_list trying to find token subset or large Jaccard
    for a in alias_list:
        a_toks = set(a.split())
        if not a_toks:
            continue
        inter = f_toks & a_toks
        # score: jaccard
        j = len(inter) / (len(f_toks | a_toks) + 1e-12)
        # treat subset highly
        if a_toks.issubset(f_toks) or f_toks.issubset(a_toks):
            j = max(j, 0.8)
        if j > best_score:
            best_score = j
            best_r = a
    if best_r and best_score >= 0.35:
        r_idx = alias_norm_to_rsg_idxs.get(best_r,[None])[0]
        if r_idx is not None:
            mapping_rows.append({
                "cnn_index": i_c, "file": crow[file_col], "file_basename": crow['file_basename'],
                "method": "token_overlap", "rsg_index": r_idx, "rsg_alias": rsg.at[r_idx,'Alias'],
                "score": float(round(best_score,3))
            })
            mapped_cnn_idx.add(i_c)
            mapped_rsg_idx.add(r_idx)

# produce mapping debug dataframe
mapping_df = pd.DataFrame(mapping_rows)
# fill helpful columns
if not mapping_df.empty:
    mapping_df['rsg_alias_norm'] = mapping_df['rsg_index'].map(lambda i: rsg.at[i,'alias_norm'])
    mapping_df['rsg_Alias'] = mapping_df['rsg_index'].map(lambda i: rsg.at[i,'Alias'])
else:
    mapping_df = pd.DataFrame(columns=["cnn_index","file","file_basename","method","rsg_index","rsg_alias","score","rsg_alias_norm","rsg_Alias"])

# unmapped CNN
all_cnn_idx = set(cnn.index.tolist())
unmapped_idx = sorted(list(all_cnn_idx - mapped_cnn_idx))
unmapped_df = cnn.loc[unmapped_idx, [file_col,'file_basename','file_norm']].copy()
unmapped_df = unmapped_df.rename(columns={file_col:'file'})

# ---- attach embeddings into RSG where mapped ----
# copy f* and img_* columns from cnn to rsg
emb_cols = [c for c in cnn.columns if str(c).startswith("f")]
img_cols = [c for c in cnn.columns if str(c).startswith("img_")]

rsg_out = rsg.copy()
# initialize columns
for c in emb_cols + img_cols + ['cnn_matched_file','cnn_matched_method','cnn_matched_score']:
    if c not in rsg_out.columns:
        rsg_out[c] = np.nan

# ---- SAFE copy embeddings from cnn -> rsg_out for mapped pairs ----
# Prepare embedding and image columns in rsg_out as floats (create if missing)
for c in emb_cols + img_cols:
    if c not in rsg_out.columns:
        rsg_out[c] = np.nan
    # coerce to float dtype if possible
    try:
        rsg_out[c] = rsg_out[c].astype(float)
    except Exception:
        # if coercion fails, replace entire column with float NaNs and continue
        rsg_out[c] = np.full(len(rsg_out), np.nan, dtype=float)

# meta columns
for meta_c in ['cnn_matched_file','cnn_matched_method','cnn_matched_score']:
    if meta_c not in rsg_out.columns:
        rsg_out[meta_c] = ""

# Now perform vectorized copy by grouping mappings by rsg index (in case duplicates)
if mapping_df.empty:
    print("No mappings to copy.")
else:
    # ensure indices are ints
    mapping_df['cnn_index'] = mapping_df['cnn_index'].astype(int)
    mapping_df['rsg_index'] = mapping_df['rsg_index'].astype(int)

    # iterate over mappings but copy rows vectorized
    for _, m in mapping_df.iterrows():
        cnn_i = int(m['cnn_index'])
        rsg_i = int(m['rsg_index'])
        try:
            # copy embedding columns (vectorized)
            if len(emb_cols) > 0:
                vals = cnn.loc[cnn_i, emb_cols].values
                # ensure scalar float array (no nested lists)
                vals = np.asarray(vals, dtype=float).ravel()
                # assign across columns
                rsg_out.loc[rsg_i, emb_cols] = vals

            # copy img_* columns
            if len(img_cols) > 0:
                vals2 = cnn.loc[cnn_i, img_cols].values
                vals2 = np.asarray(vals2, dtype=float).ravel()
                rsg_out.loc[rsg_i, img_cols] = vals2

            # copy metadata
            rsg_out.at[rsg_i, 'cnn_matched_file'] = str(cnn.at[cnn_i, file_col])
            rsg_out.at[rsg_i, 'cnn_matched_method'] = str(m.get('method', ''))
            rsg_out.at[rsg_i, 'cnn_matched_score'] = float(m.get('score', 0.0))

        except Exception as e:
            # fallback: try per-column safe assignment and log
            print(f"Warning: failed vectorized copy for cnn_i={cnn_i}, rsg_i={rsg_i}: {e}")
            for c in emb_cols:
                try:
                    rsg_out.at[rsg_i, c] = float(cnn.at[cnn_i, c])
                except Exception:
                    rsg_out.at[rsg_i, c] = np.nan
            for c in img_cols:
                try:
                    rsg_out.at[rsg_i, c] = float(cnn.at[cnn_i, c])
                except Exception:
                    rsg_out.at[rsg_i, c] = np.nan
            rsg_out.at[rsg_i, 'cnn_matched_file'] = str(cnn.at[cnn_i, file_col])
            rsg_out.at[rsg_i, 'cnn_matched_method'] = str(m.get('method', ''))
            try:
                rsg_out.at[rsg_i, 'cnn_matched_score'] = float(m.get('score', 0.0))
            except Exception:
                rsg_out.at[rsg_i, 'cnn_matched_score'] = np.nan

# After copying, count how many RSG rows have any embedding present
rsg_with_emb_count = int(rsg_out[emb_cols].notna().any(axis=1).sum())
print("Copied embeddings into RSG rows. RSG rows with embeddings now:", rsg_with_emb_count, "of", len(rsg_out))


Loading inputs...
rsg rows: 651 cnn rows: 135
Unique non-empty alias_norm count: 651
Detected file column: file
Sample cnn basenames: ['NML Cyg_g', 'NML Cyg_i', 'NML Cyg_r', 'NML Cyg_y', 'NML Cyg_z', 'S Per_g']
Copied embeddings into RSG rows. RSG rows with embeddings now: 8 of 651


In [ ]:
# run_image_only_light.py
"""
Lightweight image-only training pipeline
Inputs:
  - /content/model_training_star_level.csv    (expects embedding cols f0..fN and img_* columns)
Outputs:
  - /content/star_level_image_only_preds.csv
  - /content/sn_image_only_model.joblib
"""
import warnings, os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_SPLITS = 8
PCA_DIM = 64     # embedding dims -> PCA dims (tuneable)
RF_N_EST = 400
INPUT = "/content/model_training_star_level.csv"
OUT_PRED = "/content/star_level_image_only_preds.csv"
OUT_MODEL = "/content/sn_image_only_model.joblib"

print("Loading:", INPUT)
df = pd.read_csv(INPUT)
print("Rows:", len(df))

if "special_label" not in df.columns:
    raise SystemExit("Input must contain 'special_label' column.")

y = df["special_label"].fillna(0).astype(int).values

# detect columns: embeddings f0..fN and img_*
emb_cols = sorted([c for c in df.columns if str(c).startswith("f") and c[1:].isdigit()],
                  key=lambda x: int(x[1:]) if x[1:].isdigit() else x)
img_cols = [c for c in df.columns if c.startswith("img_")]
print("Detected embedding dims:", len(emb_cols), " img_cols:", len(img_cols))

if len(emb_cols) == 0 and len(img_cols) == 0:
    raise SystemExit("No embedding (f*) or img_* columns found. Nothing to train on.")

# Create matrices
X_emb = df[emb_cols].copy() if len(emb_cols)>0 else pd.DataFrame(index=df.index)
X_img = df[img_cols].copy() if len(img_cols)>0 else pd.DataFrame(index=df.index)

# prepare CV
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_stats = []
best_thresholds = []
oof_probs = np.zeros(len(df))
oof_preds = np.zeros(len(df))

print("\nRunning StratifiedKFold (image-only) ...")
fold = 1
for tr_idx, val_idx in skf.split(df, y):
    print("Fold", fold)
    # split arrays
    Xe_tr = X_emb.iloc[tr_idx].values if not X_emb.empty else np.zeros((len(tr_idx),0))
    Xe_val = X_emb.iloc[val_idx].values if not X_emb.empty else np.zeros((len(val_idx),0))
    Xi_tr = X_img.iloc[tr_idx].values if not X_img.empty else np.zeros((len(tr_idx),0))
    Xi_val = X_img.iloc[val_idx].values if not X_img.empty else np.zeros((len(val_idx),0))
    y_tr = y[tr_idx]; y_val = y[val_idx]

    # imputers
    imp_emb = SimpleImputer(strategy="median")
    imp_img = SimpleImputer(strategy="median") if Xi_tr.shape[1]>0 else None

    Xe_tr_imp = imp_emb.fit_transform(Xe_tr)
    Xe_val_imp = imp_emb.transform(Xe_val)
    if imp_img:
        Xi_tr_imp = imp_img.fit_transform(Xi_tr)
        Xi_val_imp = imp_img.transform(Xi_val)
    else:
        Xi_tr_imp = np.zeros((Xe_tr_imp.shape[0],0)); Xi_val_imp = np.zeros((Xe_val_imp.shape[0],0))

    # PCA on embeddings (train-only)
    if Xe_tr_imp.shape[1] > 0:
        pca = PCA(n_components=min(PCA_DIM, Xe_tr_imp.shape[1]), random_state=RANDOM_STATE)
        Xe_tr_p = pca.fit_transform(Xe_tr_imp)
        Xe_val_p = pca.transform(Xe_val_imp)
    else:
        Xe_tr_p = np.zeros((Xe_tr_imp.shape[0],0))
        Xe_val_p = np.zeros((Xe_val_imp.shape[0],0))

    # concat reduced embeddings + img stats
    Xtr = np.hstack([Xe_tr_p, Xi_tr_imp])
    Xval = np.hstack([Xe_val_p, Xi_val_imp])

    # scale
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xval_s = scaler.transform(Xval)

    # train RF (no calibration inside CV)
    rf = RandomForestClassifier(n_estimators=RF_N_EST, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(Xtr_s, y_tr)

    probs_val = rf.predict_proba(Xval_s)[:,1]
    auc = roc_auc_score(y_val, probs_val) if len(np.unique(y_val))>1 else np.nan

    # baseline @0.5
    pred50 = (probs_val >= 0.5).astype(int)
    acc50 = accuracy_score(y_val, pred50)
    prec50 = precision_score(y_val, pred50, zero_division=0)
    rec50 = recall_score(y_val, pred50, zero_division=0)
    f150 = f1_score(y_val, pred50, zero_division=0)

    # best threshold by val F1 (sweep)
    precs, recs, ths = precision_recall_curve(y_val, probs_val)
    f1s = 2 * (precs * recs) / (precs + recs + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(ths[best_idx]) if best_idx < len(ths) else 0.5
    pred_best = (probs_val >= best_thr).astype(int)
    acc_best = accuracy_score(y_val, pred_best)
    prec_best = precision_score(y_val, pred_best, zero_division=0)
    rec_best = recall_score(y_val, pred_best, zero_division=0)
    f1_best = f1_score(y_val, pred_best, zero_division=0)

    print(f" @0.5 -> acc {acc50:.3f} prec {prec50:.3f} rec {rec50:.3f} f1 {f150:.3f} auc {auc:.3f}")
    print(f" best_thr {best_thr:.4f} -> acc {acc_best:.3f} prec {prec_best:.3f} rec {rec_best:.3f} f1 {f1_best:.3f}")

    fold_stats.append((acc50, prec50, rec50, f150, auc, acc_best, prec_best, rec_best, f1_best))
    best_thresholds.append(best_thr)

    # store OOF
    oof_probs[val_idx] = probs_val
    oof_preds[val_idx] = pred_best

    fold += 1

# summarize CV
arr = np.array(fold_stats)
names = ["acc50","prec50","rec50","f150","auc","acc_best","prec_best","rec_best","f1_best"]
print("\n=== CV summary (mean ± std) ===")
for i,n in enumerate(names):
    print(f"{n:8s} : {arr[:,i].mean():.4f} ± {arr[:,i].std():.4f}")

avg_best_thr = float(np.nanmean(best_thresholds))
print("\nAverage best threshold across folds:", avg_best_thr)

# ---------- Final model on full dataset ----------
print("\nTraining final RF on full dataset (with PCA on full embeddings) ...")
# fit imputers on full data
imp_emb_full = SimpleImputer(strategy="median").fit(X_emb.values) if X_emb.shape[1]>0 else None
imp_img_full = SimpleImputer(strategy="median").fit(X_img.values) if X_img.shape[1]>0 else None

Xe_full_imp = imp_emb_full.transform(X_emb.values) if imp_emb_full is not None else np.zeros((len(df),0))
Xi_full_imp = imp_img_full.transform(X_img.values) if imp_img_full is not None else np.zeros((len(df),0))

if Xe_full_imp.shape[1] > 0:
    pca_full = PCA(n_components=min(PCA_DIM, Xe_full_imp.shape[1]), random_state=RANDOM_STATE)
    Xe_full_p = pca_full.fit_transform(Xe_full_imp)
else:
    pca_full = None
    Xe_full_p = np.zeros((len(df),0))

X_full = np.hstack([Xe_full_p, Xi_full_imp])
scaler_full = StandardScaler().fit(X_full)
X_full_s = scaler_full.transform(X_full)

final_rf = RandomForestClassifier(n_estimators=RF_N_EST, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
final_rf.fit(X_full_s, y)

# Calibrate final RF using CalibratedClassifierCV on full dataset (cv=5)
calibrator = CalibratedClassifierCV(final_rf, method="isotonic", cv=5)
calibrator.fit(X_full_s, y)
probs_full = calibrator.predict_proba(X_full_s)[:,1]

# choose threshold (use average best from CV)
threshold = avg_best_thr
preds_full = (probs_full >= threshold).astype(int)

print("\nFinal (train-eval) metrics @threshold {:.3f}:".format(threshold))
print("acc", accuracy_score(y, preds_full))
print("prec", precision_score(y, preds_full, zero_division=0))
print("rec", recall_score(y, preds_full, zero_division=0))
print("f1", f1_score(y, preds_full, zero_division=0))
try:
    print("auc", roc_auc_score(y, probs_full))
except:
    print("auc: n/a")
print("confusion:\n", confusion_matrix(y, preds_full))

# Save predictions and model bundle
out = df.copy()
out["prob_image_only"] = probs_full
out["pred_image_only"] = preds_full
out.to_csv(OUT_PRED, index=False)
print("\nWrote predictions to:", OUT_PRED)

joblib.dump({
    "imp_emb": imp_emb_full,
    "imp_img": imp_img_full,
    "pca": pca_full,
    "scaler": scaler_full,
    "calibrator": calibrator,
    "rf": final_rf,
    "emb_cols": emb_cols,
    "img_cols": img_cols,
    "threshold": threshold
}, OUT_MODEL)
print("Saved model bundle to:", OUT_MODEL)
print("\nDone.")


Loading: /content/model_training_star_level.csv
Rows: 986
Detected embedding dims: 512  img_cols: 16

Running StratifiedKFold (image-only) ...
Fold 1
 @0.5 -> acc 0.782 prec 1.000 rec 0.289 f1 0.449 auc 0.661
 best_thr 0.4240 -> acc 0.339 prec 0.317 rec 1.000 f1 0.481
Fold 2
 @0.5 -> acc 0.798 prec 1.000 rec 0.359 f1 0.528 auc 0.706
 best_thr 0.6490 -> acc 0.798 prec 1.000 rec 0.359 f1 0.528
Fold 3
 @0.5 -> acc 0.789 prec 0.929 rec 0.342 f1 0.500 auc 0.675
 best_thr 0.6663 -> acc 0.797 prec 1.000 rec 0.342 f1 0.510
Fold 4
 @0.5 -> acc 0.748 prec 0.889 rec 0.211 f1 0.340 auc 0.619
 best_thr 0.4195 -> acc 0.341 prec 0.319 rec 1.000 f1 0.484
Fold 5
 @0.5 -> acc 0.748 prec 0.818 rec 0.237 f1 0.367 auc 0.617
 best_thr 0.4212 -> acc 0.325 prec 0.314 rec 1.000 f1 0.478
Fold 6
 @0.5 -> acc 0.797 prec 0.933 rec 0.368 f1 0.528 auc 0.688
 best_thr 0.6810 -> acc 0.805 prec 1.000 rec 0.368 f1 0.538
Fold 7
 @0.5 -> acc 0.813 prec 1.000 rec 0.395 f1 0.566 auc 0.701
 best_thr 0.7197 -> acc 0.813 prec 